# TerraCascade — Flood Extent Inference (Prithvi-100M-sen1floods11)

**Goal of this notebook:** go from your downloaded Sentinel-2 scene to a `HazardEvent` flood-extent fixture (GeoJSON + metadata) that you hand to your backend teammate.

**Scene:** `S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE` (Idamalayar AOI, 10 Aug 2026)

**Steps in this notebook:**
1. Mount Drive and unzip your scene
2. Clone IBM/NASA's official demo repo (known-working code — don't reinvent the mmsegmentation setup)
3. Stack your 6 bands (Blue, Green, Red, Narrow NIR, SWIR1, SWIR2) into one aligned GeoTIFF
4. Download the pretrained checkpoint (no training needed — it's already fine-tuned)
5. Run inference → get the water/no-water mask
6. Sanity-check the mask visually
7. Convert mask → GeoJSON polygons
8. Package into the `HazardEvent` fixture JSON

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Mount Drive and unzip your scene

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import glob
matches = glob.glob('/content/drive/MyDrive/terracascade/*.zip') + \
          glob.glob('/content/drive/MyDrive/terracascade/*.SAFE')
print('Found:', matches)
SCENE_ZIP_PATH = matches[0]  # should be your one scene file

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: ['/content/drive/MyDrive/terracascade/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE.zip']


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!unzip -o -q "$SCENE_ZIP_PATH" -d /content/scene_extracted
!find /content/scene_extracted -iname "*_B02_10m.jp2" -o -iname "*_B03_10m.jp2" -o -iname "*_B04_10m.jp2" -o -iname "*_B8A_20m.jp2" -o -iname "*_B11_20m.jp2" -o -iname "*_B12_20m.jp2"

replace /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/DATASTRIP/DS_2BPS_20260810T085431_S20260810T052431/MTD_DS.xml? [y]es, [n]o, [A]ll, [N]one, [r]ename: /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R20m/T43PGM_20260810T050649_B11_20m.jp2
/content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R20m/T43PGM_20260810T050649_B8A_20m.jp2
/content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R20m/T43PGM_20260810T050649_B12_20m.jp2
/content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R10m/T43PGM_20260810T050649_B02_10m.jp2
/content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PG

**Copy the 6 paths printed above** into the cell below (edit the dict). This is scene-specific so it has to be pasted, not guessed.

In [13]:
import glob

def find_band(pattern):
    matches = glob.glob(f'/content/scene_extracted/**/{pattern}', recursive=True)
    if not matches:
        raise FileNotFoundError(f"No file found matching {pattern}")
    return matches[0]

band_paths = {
    'B02': find_band('*_B02_10m.jp2'),
    'B03': find_band('*_B03_10m.jp2'),
    'B04': find_band('*_B04_10m.jp2'),
    'B8A': find_band('*_B8A_20m.jp2'),
    'B11': find_band('*_B11_20m.jp2'),
    'B12': find_band('*_B12_20m.jp2'),
}

for band, path in band_paths.items():
    print(band, '->', path)

B02 -> /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R10m/T43PGM_20260810T050649_B02_10m.jp2
B03 -> /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R10m/T43PGM_20260810T050649_B03_10m.jp2
B04 -> /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R10m/T43PGM_20260810T050649_B04_10m.jp2
B8A -> /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R20m/T43PGM_20260810T050649_B8A_20m.jp2
B11 -> /content/scene_extracted/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE/GRANULE/L2A_T43PGM_A049237_20260810T052431/IMG_DATA/R20m/T43PGM_20260810T050649_B11_20m.jp2
B12 -> /content/scene_extracted/S2B_MSIL2A_2026081

## 2. Clone the official demo repo (known-working environment)

In [20]:
%cd /content
!rm -rf Prithvi-100M-sen1floods11-demo
!git clone https://huggingface.co/spaces/ibm-nasa-geospatial/Prithvi-100M-sen1floods11-demo
%cd /content/Prithvi-100M-sen1floods11-demo
%cd /content
!rm -rf Prithvi-100M-sen1floods11-demo
!git clone https://huggingface.co/spaces/ibm-nasa-geospatial/Prithvi-100M-sen1floods11-demo
%cd /content/Prithvi-100M-sen1floods11-demo

!pip install -q torch torchvision huggingface_hub rasterio shapely
!pip install -q -U openmim
!mim install mmcv-full
!pip install -q mmsegmentation

/content
Cloning into 'Prithvi-100M-sen1floods11-demo'...
remote: Enumerating objects: 71, done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 71 (from 1)
Receiving objects: 100% (71/71), 12.09 KiB | 12.09 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/Prithvi-100M-sen1floods11-demo
/content
Cloning into 'Prithvi-100M-sen1floods11-demo'...
remote: Enumerating objects: 71, done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 71 (from 1)
Receiving objects: 100% (71/71), 12.09 KiB | 6.05 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/Prithvi-100M-sen1floods11-demo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.0 MB/s eta 0:00:00
  Preparing meta

In [25]:
# Step 1: torch/torchvision — use oldest available compatible pair
!pip install -q torch==2.2.1+cu121 torchvision==0.17.1+cu121 --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 190.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 258.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 244.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 30.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 67.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 128.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 117.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 MB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [26]:
!pip install -q mmcv-full==1.7.1 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.2/index.html

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [24]:
# Downgrade torch to a version mmcv has prebuilt wheels for
!pip install -q torch==2.1.0+cu121 torchvision==0.16.0+cu121 --index-url https://download.pytorch.org/whl/cu121

# Install matching prebuilt mmcv-full
!pip install -q mmcv-full==1.7.1 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

# Install the OLD mmsegmentation API (0.x) that app.py actually uses
!pip install -q "mmsegmentation==0.30.0"

# Re-confirm the rest are present
!pip install -q huggingface_hub rasterio shapely

ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu121 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.0+cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.4/605.4 kB 14.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.7/831.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.8/648.8 kB 49.4 MB/s eta 0:00

In [27]:
!pip install -q "mmcv==1.7.1"
!pip install -q "mmsegmentation==0.30.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.4/605.4 kB 14.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [28]:
!pip install -q terratorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.8/643.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.8/136.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [29]:
from terratorch.registry import BACKBONE_REGISTRY

model = BACKBONE_REGISTRY.build("ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11")
print(model)

AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [30]:
pip install -q "numpy<2.1" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 93.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
terratorch 1.2.11 requires numpy>=2.2, but you have numpy 2.0.2 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 13.4.2 which is incompatible.
spopt 0.7.0 requires tqdm>=4.66.0, but you have tqdm 4.65.2 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.28.2 which is incompatible.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
tobler 0.14.0 requires tqdm>=4.67, but you have tqdm 4.65.2 which is incompatible.
yfinance 0.2.66 requires requests>=2.31, but you have requests 2.28.2 which is incompatible.
datasets 4.0.0 requires requests>=2.32.2, but you

In [ ]:
# IMPORTANT: read this before writing inference code below.
# This shows the ACTUAL working preprocessing + inference functions —
# use these, don't hand-roll your own normalization/band-order logic.
!cat app.py

## 3. Stack your 6 bands into one aligned GeoTIFF

B02/B03/B04 are natively 10m resolution; B8A/B11/B12 are natively 20m. We resample everything to a common grid (10m) before stacking, since the model needs one aligned array, not six separately-sized ones.

Band order matters: **Blue, Green, Red, Narrow NIR, SWIR1, SWIR2** — do not reorder.

In [16]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import numpy as np

band_order = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']

# Use B02 (10m) as the reference grid
with rasterio.open(band_paths['B02']) as ref:
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = (ref.height, ref.width)
    ref_profile = ref.profile

stack = np.zeros((6, *ref_shape), dtype=np.float32)

for i, band in enumerate(band_order):
    with rasterio.open(band_paths[band]) as src:
        if src.shape == ref_shape:
            stack[i] = src.read(1)
        else:
            # 20m band -> resample up to the 10m reference grid
            reproject(
                source=rasterio.band(src, 1),
                destination=stack[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=Resampling.bilinear,
            )

print('Stacked array shape (bands, height, width):', stack.shape)

# Save as a single 6-band GeoTIFF for the inference step
out_profile = ref_profile.copy()
out_profile.update(count=6, dtype='float32')

STACKED_TIF = '/content/stacked_input.tif'
with rasterio.open(STACKED_TIF, 'w', **out_profile) as dst:
    dst.write(stack)

print('Saved:', STACKED_TIF)

Stacked array shape (bands, height, width): (6, 10980, 10980)


CPLE_NotSupportedError: JP2 driver only supports creating Byte, GDT_Int16, GDT_UInt16, GDT_Int32, GDT_UInt32

**Sanity check before moving on** — plot an RGB composite. If this looks like a recognizable satellite image (not noise), band order and alignment are correct.

In [ ]:
import matplotlib.pyplot as plt

rgb = stack[[2, 1, 0]]  # Red, Green, Blue for display
rgb_norm = np.clip(rgb / np.percentile(rgb, 98), 0, 1).transpose(1, 2, 0)
plt.figure(figsize=(8, 8))
plt.imshow(rgb_norm)
plt.title('RGB sanity check — should look like a real satellite image')
plt.axis('off')
plt.show()

## 4. Download the pretrained checkpoint

This is already fine-tuned on Sen1Floods11 — you're loading weights, not training anything.

In [ ]:
from huggingface_hub import hf_hub_download

config_path = hf_hub_download(
    repo_id='ibm-nasa-geospatial/Prithvi-100M-sen1floods11',
    filename='sen1floods11_Prithvi_100M.py',
)
ckpt_path = hf_hub_download(
    repo_id='ibm-nasa-geospatial/Prithvi-100M-sen1floods11',
    filename='sen1floods11_Prithvi_100M.pth',
)
print('Config:', config_path)
print('Checkpoint:', ckpt_path)

## 5. Run inference

**This cell is a starting point, not guaranteed-correct code** — I don't have live access to verify the exact function signatures in `app.py` right now. Once you ran `!cat app.py` in step 2, adapt this cell to call whatever loading/inference functions actually appear there (they may be named differently, e.g. a custom `load_raster`/`enhance_raster_for_visualization` pair instead of raw `mmseg` calls). The general working pattern documented for this model is:


In [ ]:
from mmcv import Config
from mmseg.apis import init_segmentor, inference_segmentor

model = init_segmentor(Config.fromfile(config_path), ckpt_path, device='cuda')

# ADAPT: inference_segmentor typically expects a file path or array matching
# the preprocessing pipeline defined in the config / app.py. If this errors,
# check app.py's own inference call for the exact expected input format.
result = inference_segmentor(model, STACKED_TIF)
mask = result[0]  # 0 = no water, 1 = water, -1 = no data/cloud

print('Mask shape:', mask.shape)
print('Water pixel count:', (mask == 1).sum())

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(rgb_norm)
plt.imshow(np.ma.masked_where(mask != 1, mask), cmap='Blues', alpha=0.6)
plt.title('Water mask overlay — should trace rivers/reservoir shape, not noise')
plt.axis('off')
plt.show()

## 6. Convert mask to GeoJSON polygons

In [ ]:
import rasterio.features
import shapely.geometry
import json

water_mask = (mask == 1).astype('uint8')

shapes_gen = rasterio.features.shapes(water_mask, transform=ref_transform)
polygons = [shapely.geometry.shape(geom) for geom, val in shapes_gen if val == 1]

print(f'Found {len(polygons)} water polygons')

geojson_features = [
    {
        'type': 'Feature',
        'geometry': shapely.geometry.mapping(poly),
        'properties': {'id': f'flood_zone_{i}'},
    }
    for i, poly in enumerate(polygons)
]

flood_extent_geojson = {
    'type': 'FeatureCollection',
    'features': geojson_features,
}

with open('/content/flood_extent.geojson', 'w') as f:
    json.dump(flood_extent_geojson, f)

print('Saved: /content/flood_extent.geojson')

## 7. Package into the HazardEvent fixture

This is the file you hand directly to your backend teammate — matches the contract in `01_BACKEND.md`.

In [ ]:
hazard_event = {
    'id': 'flood-idamalayar-20260810',
    'hazard': 'flood',
    'severity': 'orange',  # set per your Orange EAP demo scenario
    'source': (
        'Prithvi-100M-sen1floods11 inference, '
        'Sentinel-2 scene S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431, '
        'Idamalayar AOI'
    ),
    'status': 'verified-demo',
    'confidence': None,  # fill in if you compute a model confidence score
    'issuedAt': '2026-08-10T05:06:49Z',
    'affectedZones': [f['properties']['id'] for f in geojson_features],
    'limitations': [
        'single-timestamp inference',
        'not a live feed',
        'demo AOI only',
    ],
}

with open('/content/hazard_event_fixture.json', 'w') as f:
    json.dump({'hazardEvent': hazard_event, 'floodExtent': flood_extent_geojson}, f, indent=2)

print('Saved: /content/hazard_event_fixture.json')
print('\nThis is the file to hand to your backend teammate.')

## Done

Download `hazard_event_fixture.json` from the Colab file browser (folder icon, left sidebar) and hand it off. If step 5 (inference) errored because `app.py`'s actual functions differ from what's here, paste the relevant part of `app.py`'s output back and we'll fix that cell to match exactly.